In [142]:
data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
!wget $data

--2025-10-31 15:33:25--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv.4’

car_fuel_efficiency 100%[===================>] 853.70K  5.06MB/s    in 0.2s    

2025-10-31 15:33:25 (5.06 MB/s) - ‘car_fuel_efficiency.csv.4’ saved [874188/874188]



In [143]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

In [144]:
df = pd.read_csv(data)
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [145]:
df.isna().sum()

,0
engine_displacement,0
num_cylinders,482
horsepower,708
vehicle_weight,0
acceleration,930
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,502


In [146]:
nan_cols = ['num_cylinders','horsepower','acceleration','num_doors']
for col in nan_cols:
  tp = df[col].dtype
  print(tp)
  df[col] = df[col].fillna(0)

float64
float64
float64
float64


In [155]:
features = ['vehicle_weight', 'model_year', 'origin', 'fuel_type']
le = LabelEncoder()
df['origin'] = le.fit_transform(df['origin'])
df['fuel_type'] = le.fit_transform(df['fuel_type'])
X = df[features]
y = df['fuel_efficiency_mpg']

model = DecisionTreeRegressor(max_depth=1, random_state=42)
model.fit(X, y)

feature_index = model.tree_.feature[0]
feature_name = features[feature_index]

print("Splitting feature:", feature_name)

Splitting feature: vehicle_weight


In [147]:
feature_columns = [
    'model_year',
    'engine_displacement',
    'vehicle_weight',
    'origin',
    'fuel_type',
    'drivetrain',
    'num_doors',
    'acceleration',
    'num_cylinders',
    'horsepower'
]

X = df[feature_columns]
y = df['fuel_efficiency_mpg']

X_train_df, X_val_df, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(X_train_df.to_dict(orient='records'))
X_val = dv.transform(X_val_df.to_dict(orient='records'))


rf = RandomForestRegressor(n_estimators=10, max_depth=10, random_state=1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"RMSE: {rmse:.3f}")

RMSE: 0.440


In [148]:
score = []

for n_est in range(10, 201, 10):
    rf = RandomForestRegressor(n_estimators=n_est, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    rmse_rounded = round(rmse, 3)
    score.append((n_est, rmse_rounded))

    print(f"n_estimators={n_est:3d} → RMSE={rmse_rounded:.3f}")


best_rmse = float('inf')
stop_at = 200

for n_est, rmse in score:
    if rmse < best_rmse:
        best_rmse = rmse
    elif rmse >= best_rmse:
        stop_at = n_est
        break

n_estimators= 10 → RMSE=0.451
n_estimators= 20 → RMSE=0.439
n_estimators= 30 → RMSE=0.436
n_estimators= 40 → RMSE=0.434
n_estimators= 50 → RMSE=0.435
n_estimators= 60 → RMSE=0.434
n_estimators= 70 → RMSE=0.433
n_estimators= 80 → RMSE=0.432
n_estimators= 90 → RMSE=0.432
n_estimators=100 → RMSE=0.432
n_estimators=110 → RMSE=0.432
n_estimators=120 → RMSE=0.432
n_estimators=130 → RMSE=0.432
n_estimators=140 → RMSE=0.432
n_estimators=150 → RMSE=0.432
n_estimators=160 → RMSE=0.432
n_estimators=170 → RMSE=0.431
n_estimators=180 → RMSE=0.431
n_estimators=190 → RMSE=0.431
n_estimators=200 → RMSE=0.431


In [149]:
max_depths = [10, 15, 20, 25]
n_est_range = range(10, 201, 10)
mean_rmse_per_depth = {}

for max_d in max_depths:
    rmse_list = []
    print(f"Testing max_depth = {max_d}")

    for n_est in n_est_range:
        rf = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=max_d,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_list.append(rmse)


    mean_rmse = np.mean(rmse_list)
    mean_rmse_per_depth[max_d] = mean_rmse
    print(f"  → Mean RMSE: {mean_rmse:.5f}")

best_max_depth = min(mean_rmse_per_depth, key=mean_rmse_per_depth.get)
print("\nMean RMSE per max_depth:")
for d, rmse in mean_rmse_per_depth.items():
    print(f"  max_depth={d}: {rmse:.5f}")

print(f"\n Best max_depth: {best_max_depth}")

Testing max_depth = 10
  → Mean RMSE: 0.43081
Testing max_depth = 15
  → Mean RMSE: 0.43376
Testing max_depth = 20
  → Mean RMSE: 0.43378
Testing max_depth = 25
  → Mean RMSE: 0.43382

Mean RMSE per max_depth:
  max_depth=10: 0.43081
  max_depth=15: 0.43376
  max_depth=20: 0.43378
  max_depth=25: 0.43382

 Best max_depth: 10


In [150]:
feature_columns = [
    'model_year',
    'engine_displacement',
    'vehicle_weight',
    'origin',
    'fuel_type',
    'drivetrain',
    'num_doors',
    'acceleration',
    'num_cylinders',
    'horsepower'
]

X = df[feature_columns]
y = df['fuel_efficiency_mpg']

X_train_df, X_val_df, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(X_train_df.to_dict(orient='records'))
X_val = dv.transform(X_val_df.to_dict(orient='records'))


rf = RandomForestRegressor(n_estimators=10, max_depth=10, random_state=1)
rf.fit(X_train, y_train)
rf.fit(X_train, y_train)


RandomForestRegressor(max_depth=10, n_estimators=10, random_state=1)

In [151]:
feature_names = dv.get_feature_names_out()

importances = rf.feature_importances_

feature_importance_list = sorted(
    zip(feature_names, importances),
    key=lambda x: x[1],
    reverse=True
)

print("Top Feature Importances:")
for name, imp in feature_importance_list[:10]:
    print(f"{name}: {imp:.4f}")

Top Feature Importances:
vehicle_weight: 0.9697
horsepower: 0.0140
acceleration: 0.0099
engine_displacement: 0.0017
model_year: 0.0017
num_cylinders: 0.0011
num_doors: 0.0008
origin=USA: 0.0002
origin=Asia: 0.0002
origin=Europe: 0.0002


In [152]:
import xgboost as xgb
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

xgb_params_03 = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_03 = xgb.train(
    xgb_params_03,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)


y_pred_03 = model_03.predict(dval)
rmse_03 = np.sqrt(mean_squared_error(y_val, y_pred_03))
print(f"RMSE (eta=0.3): {rmse_03:.5f}")

xgb_params_01 = xgb_params_03.copy()
xgb_params_01['eta'] = 0.1

model_01 = xgb.train(
    xgb_params_01,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=False
)

y_pred_01 = model_01.predict(dval)
rmse_01 = np.sqrt(mean_squared_error(y_val, y_pred_01))
print(f"RMSE (eta=0.1): {rmse_01:.5f}")
if rmse_01 < rmse_03:
    print("Best: eta=0.1")
elif rmse_03 < rmse_01:
    print("Best: eta=0.3")
else:
    print("Both give equal value")

RMSE (eta=0.3): 0.43920
RMSE (eta=0.1): 0.41524
Best: eta=0.1
